In [ ]:
%pip install anthropic python-dotenv

In [3]:
#Load env variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [4]:
#Create an API client
from anthropic import Anthropic
import os

client = Anthropic()
model = os.environ["ANTHROPIC_MODEL"]
print(model)

anthropic.claude-4-5-haiku


In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)
    
def chat(messages, system_prompt=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 5000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system_prompt:
        params["system"] = system_prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
  
    message = client.messages.create(**params)
    return message.content[0].text

O modelo claude-4-6-sonnet via AWS Bedrock não suporta a técnica de "prefill" (pré-popular a última mensagem do assistente) usada abaixo. Essa funcionalidade só funciona na API direta da Anthropic.

Alternativa: usar haiku ou um system_prompt instruindo o modelo a responder apenas com JSON

In [7]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
print(text)


{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}

